In [1]:
import pandas as pd
crimedf = pd.read_csv("usarrests.csv")

In [2]:
crimedf.shape

(50, 4)

In [3]:
crimedf.head()

,Unnamed: 0,Murder,Assault,UrbanPop
0,Alabama,13.2,236.0,58
1,Alaska,10.0,263.0,48
2,Arizona,8.1,294.0,80
3,Arkansas,8.8,190.0,50
4,California,9.0,276.0,91


In [4]:
crimedf.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  50 non-null     str    
 1   Murder      50 non-null     float64
 2   Assault     49 non-null     float64
 3   UrbanPop    50 non-null     int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 1.7 KB


In [5]:
round(crimedf.describe(),3)

,Murder,Assault,UrbanPop
count,50.000,49.000,50.000
mean,7.788,182.184,74.200
std,4.356,130.877,73.408
min,0.800,45.000,6.000
25%,4.075,109.000,53.250
50%,7.250,159.000,66.000
75%,11.250,249.000,77.750
max,17.400,879.000,570.000


In [6]:
crimedf.duplicated().sum()

np.int64(0)

At a glance, we can see a few suspect values.

## Suspicious arrest data points
- Assault only has a count of 49, so a row is missing this value
- UrbanPop maximum is 570, which is impossible
- Assault maximum is 879, which is extreme but possible
- Murder minimum is 0.800, which is extreme but possible

There are no duplicated rows.

In [7]:
crimedf.isna().sum()

Unnamed: 0    0
Murder        0
Assault       1
UrbanPop      0
dtype: int64

From the above summary, we can see that a single row is missing a datapoint for the "Assault" column. Knowing that double brackets allow us to see a subset of information as a new dataframe, let us see what that row looks like.

In [8]:
crimedf[ crimedf["Assault"].isna() ]

,Unnamed: 0,Murder,Assault,UrbanPop
9,Georgia,17.4,NaN,60


Georgia is the row missing an "Assault" value. However, it does at least contain information for "Murder" and "UrbanPop". As a frame of comparison, let us see which states have comparable values for Urban Population (%). A range of +-2% is deemed reasonable.

In [9]:
crimedf[ (crimedf["UrbanPop"] >= 58) & (crimedf["UrbanPop"] <= 62) ]

,Unnamed: 0,Murder,Assault,UrbanPop
0,Alabama,13.2,236.0,58
9,Georgia,17.4,NaN,60
26,Nebraska,4.3,102.0,62
41,Tennessee,13.2,188.0,59
49,Wyoming,6.8,161.0,60


Four other states have similar UrbanPop values, but that is not the full picture: A state could have a very small total population (arbitrarily say 2,000) that is incomparable to Georgia's possible total population (say 2million, again arbitrarily), but they might be distributed similarly, ie. with 40% of the population living outside urban centers. So let us now assume that as total population size increases, both Assault and Murder increase at comparable rates. With that in mind, let us narrow our results further to states that have similar UrbanPop and Murder values as Georgia.

In [10]:
crimedf[ (crimedf["UrbanPop"] >= 58) & (crimedf["UrbanPop"] <= 62) & (crimedf["Murder"] > 10) ]

,Unnamed: 0,Murder,Assault,UrbanPop
0,Alabama,13.2,236.0,58
9,Georgia,17.4,NaN,60
41,Tennessee,13.2,188.0,59


We now see that Alabama and Tennessee are the most similar to Georgia in Murder and UrbanPop. Bear in mind that our goal is to rectify the row that is missing an Assault value. Because there are only fifty rows total, eliminating Georgia entirely would create too much of a gap in the data. Instead, we will use the mean of Alabama and Tennessee's Assault values to impute the missing data.

In [11]:
crime60df = crimedf[ (crimedf["UrbanPop"] >= 58) & (crimedf["UrbanPop"] <= 62) & (crimedf["Murder"] > 10) ]

In [12]:
crime60df

,Unnamed: 0,Murder,Assault,UrbanPop
0,Alabama,13.2,236.0,58
9,Georgia,17.4,NaN,60
41,Tennessee,13.2,188.0,59


In [13]:
round(crime60df.describe(),3)

,Murder,Assault,UrbanPop
count,3.000,2.000,3.0
mean,14.600,212.000,59.0
std,2.425,33.941,1.0
min,13.200,188.000,58.0
25%,13.200,200.000,58.5
50%,13.200,212.000,59.0
75%,15.300,224.000,59.5
max,17.400,236.000,60.0


In [14]:
georgia_fill_value = crime60df["Assault"].mean()
crimedf["Assault"] = crimedf["Assault"].fillna(georgia_fill_value)
print(georgia_fill_value)

212.0


The mean of Tennessee and Alabama's Assault values is 212. The next step is to verify that this mean was imputed as Georgia's Assault value.

In [15]:
crimedf.isna().sum()

Unnamed: 0    0
Murder        0
Assault       0
UrbanPop      0
dtype: int64

In [16]:
crimedf[ crimedf["UrbanPop"] == 60 ]

,Unnamed: 0,Murder,Assault,UrbanPop
9,Georgia,17.4,212.0,60
49,Wyoming,6.8,161.0,60


Now we return to the overall dataframe to search for other suspicious values.

In [17]:
round(crimedf.describe(),3)

,Murder,Assault,UrbanPop
count,50.000,50.000,50.000
mean,7.788,182.780,74.200
std,4.356,129.604,73.408
min,0.800,45.000,6.000
25%,4.075,109.000,53.250
50%,7.250,159.000,66.000
75%,11.250,249.000,77.750
max,17.400,879.000,570.000


The maximum UrbanPop value is 570%, which is not possible because population distribution cannot exceed 100%. Let us view the row(s) in question.

In [18]:
crimedf[ crimedf["UrbanPop"] > 100 ]

,Unnamed: 0,Murder,Assault,UrbanPop
14,Iowa,2.2,56.0,570


There is only a single row, Iowa, that has an impossible UrbanPop value. It can be reasonably assumed that "570" is a typo and the intended value may have been 57, which would be between 25% and 50% of the range. We will correct the error by replacing the existing value with 57.

In [19]:
crimedf["UrbanPop"] = crimedf["UrbanPop"].replace(570, 57)

In [20]:
round(crimedf.describe(),3)

,Murder,Assault,UrbanPop
count,50.000,50.000,50.000
mean,7.788,182.780,63.940
std,4.356,129.604,16.453
min,0.800,45.000,6.000
25%,4.075,109.000,53.250
50%,7.250,159.000,66.000
75%,11.250,249.000,76.500
max,17.400,879.000,91.000


With the new summary table, we can see that there are no longer any UrbanPop values above 100.

We look at the next suspicious value, the maximum Assault value of 879. While this is not impossible, it seems unlikely. Let us view the suspicious row in more detail.

In [21]:
crimedf [ crimedf ["Assault"] > 300 ]

,Unnamed: 0,Murder,Assault,UrbanPop
8,Florida,15.4,335.0,80
32,North Carolina,13.0,337.0,45
39,South Carolina,14.4,879.0,48


South Carolina's Assault value is an extreme outlier, but its Murder and UrbanPop values are not. If we continue our earlier assumption that Murder and UrbanPop likely increase at comparable rates, 879 does not seem reliable. Following a similar approach to Georgia, we will find comparable rows.

In [22]:
crimedf [ (crimedf ["UrbanPop"] >= 45) & (crimedf ["UrbanPop"] <= 50) ]

,Unnamed: 0,Murder,Assault,UrbanPop
1,Alaska,10.0,263.0,48
3,Arkansas,8.8,190.0,50
32,North Carolina,13.0,337.0,45
39,South Carolina,14.4,879.0,48
40,South Dakota,3.8,86.0,45


North Carolina and Alaska have similar Murder and UrbanPop values, so it is reasonable to imagine that South Carolina's Assault value is in their ballpark. That said, we do not have a way to predict exactly what the correct value should be, and it is not a clear decimal typo--unlike Iowa's UrbanPop issue, 879.0 is not necessarily a typo of 87.9. The best thing to do at this stage is to flag the row as potentially containing an error.

Lastly, we turn to the 0.800 minimum Murder value noted earlier. Let us look at whether it is especially unusual by looking at everything in the bottom 25% of the range.

In [23]:
crimedf [ crimedf ["Murder"] <= 4.075 ]

,Unnamed: 0,Murder,Assault,UrbanPop
6,Connecticut,3.3,110.0,77
11,Idaho,2.6,120.0,54
14,Iowa,2.2,56.0,57
18,Maine,2.1,83.0,51
22,Minnesota,2.7,72.0,66
28,New Hampshire,2.1,57.0,56
33,North Dakota,0.8,45.0,44
38,Rhode Island,3.4,174.0,87
40,South Dakota,3.8,86.0,45
43,Utah,3.2,120.0,80


It appears that North Dakota, which has a Murder value of 0.8, is the only row with a Murder value lower than 2.1; New Hampshire and Maine are North Dakota's closest neighbors in this column. While 0.8 does seem unusually low, it is not so far and away from 2.1 that it is an obvious outlier or error. The most appropriate action here is to flag it as suspect but not alter the dataset.

Now we look at the basic features of the data after cleaning, then save it as a new .csv file.

In [24]:
crimedf.head()

,Unnamed: 0,Murder,Assault,UrbanPop
0,Alabama,13.2,236.0,58
1,Alaska,10.0,263.0,48
2,Arizona,8.1,294.0,80
3,Arkansas,8.8,190.0,50
4,California,9.0,276.0,91


In [25]:
crimedf.shape

(50, 4)

In [26]:
crimedf.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  50 non-null     str    
 1   Murder      50 non-null     float64
 2   Assault     50 non-null     float64
 3   UrbanPop    50 non-null     int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 1.7 KB


In [27]:
round(crimedf.describe(),3)

,Murder,Assault,UrbanPop
count,50.000,50.000,50.000
mean,7.788,182.780,63.940
std,4.356,129.604,16.453
min,0.800,45.000,6.000
25%,4.075,109.000,53.250
50%,7.250,159.000,66.000
75%,11.250,249.000,76.500
max,17.400,879.000,91.000


In [28]:
crimedf.to_csv("arrests_clean.csv", index=False)

I consulted the pandas User Guide through-out this lab.<p>
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html#pandas.DataFrame.replace

Github Repo URL: https://github.com/journeyrahman/cs82a-portfolio